# Prédiction Multi-Label de la Toxicité des Molécules
## Dataset TOX21 | 2063 Features | 12 Cibles Simultanées

---

### Architecture du pipeline
```
SMILES  ──►  RDKit  ──►  15 descripteurs + 2048 bits Morgan
                         (2063 features au total)
                    ──►  MultiOutputClassifier
                         ├── RandomForest  × 12
                         ├── LogisticReg   × 12
                         └── XGBoost       × 12
                    ──►  Évaluation multi-label
                         (AUC-ROC, F1, Hamming, Jaccard)
                    ──►  SHAP
                    ──►  Profils de toxicité
```

### Cibles TOX21
| Groupe | Labels |
|--------|--------|
| **Nuclear Receptors** | NR-AR, NR-AR-LBD, NR-AhR, NR-Aromatase, NR-ER, NR-ER-LBD, NR-PPAR-gamma |
| **Stress Response**   | SR-ARE, SR-ATAD5, SR-HSE, SR-MMP, SR-p53 |


In [1]:
# ===================================================================
# 0. IMPORTS & CONFIGURATION
# ===================================================================
import os, sys, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # pas de fenetre popup - compatible VS Code
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, DataStructs, AllChem
from rdkit import RDLogger

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, hamming_loss, jaccard_score,
    confusion_matrix, roc_curve, auc,
    average_precision_score, classification_report,
    precision_score, recall_score
)

from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
RDLogger.DisableLog('rdApp.*')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# ── Chemins absolus (independants du dossier de lancement Jupyter) ──
# VS Code lance Jupyter depuis la racine du projet (pas depuis notebooks/)
_cwd = os.path.abspath('')
if os.path.basename(_cwd).lower() == 'notebooks':
    PROJECT_ROOT = os.path.dirname(_cwd)
else:
    PROJECT_ROOT = _cwd

DATA_PATH   = os.path.join(PROJECT_ROOT, 'data',   'tox21.csv')
MODEL_PATH  = os.path.join(PROJECT_ROOT, 'models', 'multilabel_rf_model.pkl')
FIG_DIR     = os.path.join(PROJECT_ROOT, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(os.path.join(PROJECT_ROOT, 'models'), exist_ok=True)

# ── Constantes ──────────────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.20
FP_RADIUS    = 2
FP_NBITS     = 2048

TARGET_COLS = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase',
    'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
    'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'
]

DESCRIPTOR_NAMES = [
    'MolWt', 'LogP', 'TPSA', 'NumAtoms', 'NumBonds',
    'HBondDonors', 'HBondAcceptors', 'RotatableBonds',
    'AromaticRings', 'FractionCSP3', 'HeavyAtomCount',
    'NumRings', 'NumHeteroatoms', 'NumAliphaticRings', 'MolMR'
]
FEATURE_NAMES = DESCRIPTOR_NAMES + [f'FP_{i}' for i in range(FP_NBITS)]

print('Configuration terminee')
print(f'   Racine projet : {PROJECT_ROOT}')
print(f'   Dataset       : {DATA_PATH}')
print(f'   Existe        : {os.path.exists(DATA_PATH)}')
print(f'   {len(TARGET_COLS)} cibles  |  {len(DESCRIPTOR_NAMES)} descripteurs  |  {FP_NBITS} bits Morgan')


c:\Users\oumaimaaa\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration terminee
   Racine projet : c:\Users\oumaimaaa\Desktop\master big data\S2 BDDS\Machine learning\PROJET MACHING LEARNING
   Dataset       : c:\Users\oumaimaaa\Desktop\master big data\S2 BDDS\Machine learning\PROJET MACHING LEARNING\data\tox21.csv
   Existe        : True
   12 cibles  |  15 descripteurs  |  2048 bits Morgan


---
## 1. Chargement des Données


In [2]:
df = pd.read_csv(DATA_PATH)
print(f'Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Colonnes   : {list(df.columns)}')
df.head()


Dimensions : 7,831 lignes × 14 colonnes
Colonnes   : ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53', 'mol_id', 'smiles']


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


---
## 2. EDA — Analyse Exploratoire


In [3]:
# ── 2.1 Statistiques générales ───────────────────────────────────────────────
print('='*65)
print('STATISTIQUES DES 12 CIBLES DE TOXICITÉ')
print('='*65)
stats = []
for t in TARGET_COLS:
    col = df[t].dropna()
    n1 = int((col == 1.0).sum()); n0 = int((col == 0.0).sum())
    stats.append({'Cible': t, 'N valide': len(col), 'N positifs': n1, 'N négatifs': n0,
                  '% positifs': round(100*n1/len(col), 1),
                  'Manquants': df[t].isna().sum(), '% manquants': round(100*df[t].isna().sum()/len(df),1)})
stats_df = pd.DataFrame(stats).set_index('Cible')
stats_df


STATISTIQUES DES 12 CIBLES DE TOXICITÉ


,N valide,N positifs,N négatifs,% positifs,Manquants,% manquants
Cible,,,,,,
NR-AR,7265,309,6956,4.3,566,7.2
NR-AR-LBD,6758,237,6521,3.5,1073,13.7
NR-AhR,6549,768,5781,11.7,1282,16.4
NR-Aromatase,5821,300,5521,5.2,2010,25.7
NR-ER,6193,793,5400,12.8,1638,20.9
NR-ER-LBD,6955,350,6605,5.0,876,11.2
NR-PPAR-gamma,6450,186,6264,2.9,1381,17.6
SR-ARE,5832,942,4890,16.2,1999,25.5
SR-ATAD5,7072,264,6808,3.7,759,9.7


In [4]:
# ── 2.2 Carte des valeurs manquantes ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(df[TARGET_COLS].isnull().T, ax=axes[0], cmap='RdYlGn_r',
            cbar_kws={'label': 'Manquant'}, yticklabels=TARGET_COLS, xticklabels=False)
axes[0].set_title('Carte des Valeurs Manquantes', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Molécules'); axes[0].set_ylabel('Cibles')

pct_miss = (df[TARGET_COLS].isnull().sum() / len(df) * 100).sort_values(ascending=True)
bars = axes[1].barh(pct_miss.index, pct_miss.values,
                    color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(TARGET_COLS))))
for b, v in zip(bars, pct_miss.values):
    axes[1].text(v+0.3, b.get_y()+b.get_height()/2, f'{v:.1f}%', va='center', fontsize=9)
axes[1].set_xlabel('% manquants')
axes[1].set_title('% Valeurs Manquantes par Cible', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/01_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()


In [5]:
# ── 2.3 Distribution des classes ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.ravel()
for i, t in enumerate(TARGET_COLS):
    c  = df[t].value_counts(dropna=True)
    n0, n1 = int(c.get(0,0)), int(c.get(1,0))
    pct = 100*n1/(n0+n1) if (n0+n1) else 0
    bars = axes[i].bar(['Négatif', 'Positif'], [n0, n1], color=['#2196F3','#F44336'], edgecolor='white')
    for b in bars:
        h = b.get_height()
        axes[i].text(b.get_x()+b.get_width()/2, h+5, f'{h:,}', ha='center', va='bottom', fontsize=8)
    axes[i].set_title(f'{t}\n(+:{pct:.1f}%)', fontsize=9, fontweight='bold')
plt.suptitle('Distribution des Classes par Cible (TOX21)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/02_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [6]:
# ── 2.4 Taux de positivité ───────────────────────────────────────────────────
pos_s = pd.Series({t: 100*(df[t]==1).sum()/(df[t].dropna().__len__()) for t in TARGET_COLS}).sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(pos_s.index, pos_s.values,
               color=plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(pos_s))))
ax.axvline(pos_s.mean(), color='navy', ls='--', label=f'Moyenne : {pos_s.mean():.1f}%')
for b, v in zip(bars, pos_s.values):
    ax.text(v+0.1, b.get_y()+b.get_height()/2, f'{v:.1f}%', va='center', fontsize=9)
ax.set_xlabel('% molécules toxiques')
ax.set_title('Déséquilibre des Classes par Cible', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/03_positive_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Cible la plus déséquilibrée : {pos_s.idxmin()} ({pos_s.min():.1f}%)')
print(f'Cible la moins déséquilibrée: {pos_s.idxmax()} ({pos_s.max():.1f}%)')


Cible la plus déséquilibrée : NR-PPAR-gamma (2.9%)
Cible la moins déséquilibrée: SR-ARE (16.2%)


In [7]:
# ── 2.5 Corrélation entre labels ────────────────────────────────────────────
df_comp_eda = df[TARGET_COLS].dropna()
corr = df_comp_eda.corr()
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Corrélation entre les 12 Cibles', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/04_label_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


In [8]:
# ── 2.6 Co-occurrence des labels positifs ───────────────────────────────────
cooc = df_comp_eda.T.dot(df_comp_eda).astype(int)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cooc, ax=ax, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5)
ax.set_title('Co-occurrence des Labels Positifs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/05_label_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 3. Prétraitement


In [9]:
# Filtre SMILES invalides + doublons + NaN labels
def is_valid(smi):
    try: return Chem.MolFromSmiles(str(smi)) is not None
    except: return False

df = df[df['smiles'].apply(is_valid)].copy()
df = df.drop_duplicates(subset=['smiles']).copy()
df[TARGET_COLS] = df[TARGET_COLS].apply(pd.to_numeric, errors='coerce')
df_complete = df.dropna(subset=TARGET_COLS).copy()

print(f'Dataset original    : {df.shape[0]:,} molécules')
print(f'Dataset complet     : {len(df_complete):,} molécules (12 labels non-NaN)')

print('\nDistribution finale des labels :')
for t in TARGET_COLS:
    n1 = int((df_complete[t]==1).sum())
    n0 = int((df_complete[t]==0).sum())
    print(f'  {t:<18}: {n1:4d} positifs / {n0:4d} négatifs  (ratio {n0//max(n1,1):2d}:1)')


Dataset original    : 7,823 molécules
Dataset complet     : 3,074 molécules (12 labels non-NaN)

Distribution finale des labels :
  NR-AR             :   59 positifs / 3015 négatifs  (ratio 51:1)
  NR-AR-LBD         :   34 positifs / 3040 négatifs  (ratio 89:1)
  NR-AhR            :  154 positifs / 2920 négatifs  (ratio 18:1)
  NR-Aromatase      :   55 positifs / 3019 négatifs  (ratio 54:1)
  NR-ER             :  242 positifs / 2832 négatifs  (ratio 11:1)
  NR-ER-LBD         :   66 positifs / 3008 négatifs  (ratio 45:1)
  NR-PPAR-gamma     :   26 positifs / 3048 négatifs  (ratio 117:1)
  SR-ARE            :  196 positifs / 2878 négatifs  (ratio 14:1)
  SR-ATAD5          :   11 positifs / 3063 négatifs  (ratio 278:1)
  SR-HSE            :   50 positifs / 3024 négatifs  (ratio 60:1)
  SR-MMP            :  142 positifs / 2932 négatifs  (ratio 20:1)
  SR-p53            :   28 positifs / 3046 négatifs  (ratio 108:1)


---
## 4. Feature Engineering

Chaque molécule → **2 063 features** :
- **15 descripteurs physicochimiques** (RDKit) : MolWt, LogP, TPSA, HBD, HBA…
- **2048 bits** Morgan Fingerprint radius=2 (ECFP4)


In [10]:
# ── Fonctions de featurisation ───────────────────────────────────────────────
def compute_descriptors(mol):
    try:
        return [
            Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
            mol.GetNumAtoms(), mol.GetNumBonds(),
            rdMolDescriptors.CalcNumHBD(mol), rdMolDescriptors.CalcNumHBA(mol),
            rdMolDescriptors.CalcNumRotatableBonds(mol),
            rdMolDescriptors.CalcNumAromaticRings(mol),
            rdMolDescriptors.CalcFractionCSP3(mol),
            mol.GetNumHeavyAtoms(), rdMolDescriptors.CalcNumRings(mol),
            rdMolDescriptors.CalcNumHeteroatoms(mol),
            rdMolDescriptors.CalcNumAliphaticRings(mol), Descriptors.MolMR(mol),
        ]
    except: return None

def compute_morgan_fp(mol):
    fp  = AllChem.GetMorganFingerprintAsBitVect(mol, FP_RADIUS, nBits=FP_NBITS)
    arr = np.zeros(FP_NBITS, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

# ── Extraction ───────────────────────────────────────────────────────────────
print('Extraction des features (patientez 1-2 min)...')
descs, fps, valid_idx = [], [], []
for idx, row in df_complete.iterrows():
    mol  = Chem.MolFromSmiles(row['smiles'])
    if mol is None: continue
    desc = compute_descriptors(mol)
    if desc is None: continue
    descs.append(desc)
    fps.append(compute_morgan_fp(mol))
    valid_idx.append(idx)

X = np.hstack([np.array(descs, np.float32), np.array(fps, np.float32)])
y = df_complete.loc[valid_idx, TARGET_COLS].values.astype(int)

print(f'✅  X : {X.shape}   y : {y.shape}')
print(f'    = {len(DESCRIPTOR_NAMES)} descripteurs + {FP_NBITS} bits Morgan = {X.shape[1]} features')


Extraction des features (patientez 1-2 min)...
✅  X : (3074, 2063)   y : (3074, 12)
    = 15 descripteurs + 2048 bits Morgan = 2063 features


In [11]:
# Statistiques des descripteurs
desc_df = pd.DataFrame(np.array(descs), columns=DESCRIPTOR_NAMES)
desc_df.describe().round(2)


,MolWt,LogP,TPSA,NumAtoms,NumBonds,HBondDonors,HBondAcceptors,RotatableBonds,AromaticRings,FractionCSP3,HeavyAtomCount,NumRings,NumHeteroatoms,NumAliphaticRings,MolMR
count,3074.00,3074.00,3074.00,3074.00,3074.00,3074.00,3074.00,3074.00,3074.00,3074.00,3074.00,3074.00,3074.00,3074.00,3074.00
mean,217.14,1.81,50.56,14.54,14.67,1.01,2.86,3.70,0.76,0.50,14.54,1.18,4.08,0.42,56.71
std,120.93,2.05,45.34,8.09,8.94,1.56,2.48,3.93,0.86,0.34,8.09,1.40,3.19,1.13,30.10
min,32.04,-17.41,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00
25%,138.71,0.74,20.31,9.00,9.00,0.00,1.00,1.00,0.00,0.20,9.00,0.00,2.00,0.00,37.08
50%,186.35,1.79,40.46,13.00,12.00,1.00,2.00,3.00,1.00,0.50,13.00,1.00,3.00,0.00,49.54
75%,266.29,2.84,66.91,18.00,18.00,2.00,4.00,5.00,1.00,0.83,18.00,2.00,5.00,1.00,70.29
max,1550.19,13.05,633.20,88.00,96.00,24.00,40.00,33.00,7.00,1.00,88.00,30.00,40.00,30.00,304.49


In [12]:
# Distribution des descripteurs
fig, axes = plt.subplots(3, 5, figsize=(20, 10))
axes = axes.ravel()
for i, name in enumerate(DESCRIPTOR_NAMES):
    axes[i].hist(np.array(descs)[:,i], bins=40, color='#1976D2', alpha=0.8, edgecolor='white')
    axes[i].set_title(name, fontsize=9, fontweight='bold')
plt.suptitle('Distribution des 15 Descripteurs Moléculaires', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/06_descriptor_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [13]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f'Train : {X_train.shape[0]:,} molécules')
print(f'Test  : {X_test.shape[0]:,} molécules')
print(f'Ratio : {100*(1-TEST_SIZE):.0f}/{100*TEST_SIZE:.0f}')


Train : 2,459 molécules
Test  : 615 molécules
Ratio : 80/20


---
## 5. Modélisation Multi-Label

> **Stratégie** : `MultiOutputClassifier` → entraîne un classifieur binaire indépendant par cible.


In [14]:
# ── 5.1 Random Forest ────────────────────────────────────────────────────────
print('[1/3] Entraînement Random Forest...')
rf_model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=200, min_samples_split=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ), n_jobs=-1
)
rf_model.fit(X_train, y_train)
print('✅ Random Forest entraîné  (200 arbres × 12 cibles)')


[1/3] Entraînement Random Forest...
✅ Random Forest entraîné  (200 arbres × 12 cibles)


In [15]:
# ── 5.2 Logistic Regression ──────────────────────────────────────────────────
print('[2/3] Mise à l\'échelle + Logistic Regression...')
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr_model = MultiOutputClassifier(
    LogisticRegression(
        C=1.0, max_iter=1000, class_weight='balanced',
        solver='lbfgs', random_state=RANDOM_STATE, n_jobs=-1
    ), n_jobs=-1
)
lr_model.fit(X_train_sc, y_train)
print('✅ Logistic Regression entraîné')


[2/3] Mise à l'échelle + Logistic Regression...


✅ Logistic Regression entraîné


In [16]:
# ── 5.3 XGBoost ──────────────────────────────────────────────────────────────
print('[3/3] Entraînement XGBoost...')
xgb_model = MultiOutputClassifier(
    xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    ), n_jobs=-1
)
xgb_model.fit(X_train, y_train)
print('✅ XGBoost entraîné')


[3/3] Entraînement XGBoost...
✅ XGBoost entraîné


---
## 6. Évaluation Multi-Label


In [17]:
def evaluate(model, X, y_true, name='', scaler=None):
    X_in    = scaler.transform(X) if scaler else X
    y_pred  = model.predict(X_in)
    y_proba = np.column_stack([est.predict_proba(X_in)[:,1] for est in model.estimators_])
    per = []
    for i, t in enumerate(TARGET_COLS):
        try:    auc_s = roc_auc_score(y_true[:,i], y_proba[:,i])
        except: auc_s = float('nan')
        try:    ap = average_precision_score(y_true[:,i], y_proba[:,i])
        except: ap = float('nan')
        per.append({'Cible': t, 'AUC-ROC': round(auc_s,4), 'Avg Prec.': round(ap,4),
                    'F1': round(f1_score(y_true[:,i],y_pred[:,i],zero_division=0),4),
                    'Positifs': int(y_true[:,i].sum())})
    pl = pd.DataFrame(per).set_index('Cible')
    return {'name': name, 'y_pred': y_pred, 'y_proba': y_proba,
            'hamming':   round(hamming_loss(y_true, y_pred), 4),
            'jaccard':   round(jaccard_score(y_true, y_pred, average='macro', zero_division=0), 4),
            'f1_micro':  round(f1_score(y_true, y_pred, average='micro',  zero_division=0), 4),
            'f1_macro':  round(f1_score(y_true, y_pred, average='macro',  zero_division=0), 4),
            'auc_macro': round(float(np.nanmean(pl['AUC-ROC'])), 4),
            'per_label': pl}

print('Évaluation en cours...')
rf_res  = evaluate(rf_model,  X_test, y_test, 'Random Forest')
lr_res  = evaluate(lr_model,  X_test, y_test, 'Logistic Regression', scaler=scaler)
xgb_res = evaluate(xgb_model, X_test, y_test, 'XGBoost')
print('✅ Évaluation terminée')


Évaluation en cours...
✅ Évaluation terminée


In [18]:
# ── Tableau comparatif ───────────────────────────────────────────────────────
comp_df = pd.DataFrame([
    {'Modèle': r['name'], 'Hamming Loss ↓': r['hamming'], 'Jaccard (macro)': r['jaccard'],
     'F1 (micro)': r['f1_micro'], 'F1 (macro)': r['f1_macro'], 'AUC-ROC (macro)': r['auc_macro']}
    for r in [rf_res, lr_res, xgb_res]
]).set_index('Modèle')

print('='*65)
print('     COMPARAISON DES MODÈLES MULTI-LABEL (test set)')
print('='*65)
print(comp_df.to_string())
best = comp_df['AUC-ROC (macro)'].idxmax()
print(f'\n🏆 Meilleur modèle : {best}  (AUC = {comp_df.loc[best,"AUC-ROC (macro)"]})')


     COMPARAISON DES MODÈLES MULTI-LABEL (test set)
                     Hamming Loss ↓  Jaccard (macro)  F1 (micro)  F1 (macro)  AUC-ROC (macro)
Modèle                                                                                       
Random Forest                0.0305           0.1477      0.2525      0.2371           0.7833
Logistic Regression          0.0531           0.1490      0.2314      0.2483           0.6934
XGBoost                      0.0308           0.1350      0.2305      0.2127           0.7649

🏆 Meilleur modèle : Random Forest  (AUC = 0.7833)


In [19]:
# ── AUC-ROC par cible (tous modèles) ────────────────────────────────────────
auc_table = pd.DataFrame({
    'Random Forest':       rf_res['per_label']['AUC-ROC'],
    'Logistic Regression': lr_res['per_label']['AUC-ROC'],
    'XGBoost':             xgb_res['per_label']['AUC-ROC'],
})
auc_table['Meilleur'] = auc_table.idxmax(axis=1)
auc_table


,Random Forest,Logistic Regression,XGBoost,Meilleur
Cible,,,,
NR-AR,0.6860,0.5461,0.6665,Random Forest
NR-AR-LBD,0.7029,0.6148,0.7292,XGBoost
NR-AhR,0.8145,0.6864,0.7769,Random Forest
NR-Aromatase,0.7713,0.6702,0.8217,XGBoost
NR-ER,0.6029,0.5670,0.5516,Random Forest
NR-ER-LBD,0.7852,0.7217,0.7173,Random Forest
NR-PPAR-gamma,0.9112,0.8230,0.9515,XGBoost
SR-ARE,0.6917,0.4788,0.7134,XGBoost
SR-ATAD5,0.9700,0.9488,0.8742,Random Forest


In [20]:
# ── Visualisation comparaison ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Métriques globales
metrics_keys = ['f1_micro', 'f1_macro', 'auc_macro', 'jaccard']
metrics_lbls = ['F1 micro', 'F1 macro', 'AUC-ROC', 'Jaccard']
x, w = np.arange(4), 0.25
colors_m = ['#1976D2', '#388E3C', '#F57C00']
for j, res in enumerate([rf_res, lr_res, xgb_res]):
    axes[0].bar(x + j*w, [res[k] for k in metrics_keys], w,
                label=res['name'], color=colors_m[j], alpha=0.85)
axes[0].set_xticks(x+w); axes[0].set_xticklabels(metrics_lbls, rotation=15)
axes[0].set_ylim([0,1]); axes[0].set_title('Métriques Globales', fontsize=12, fontweight='bold')
axes[0].legend()

# AUC par cible
x2 = np.arange(len(TARGET_COLS))
for j, res in enumerate([rf_res, lr_res, xgb_res]):
    axes[1].bar(x2 + j*w, res['per_label']['AUC-ROC'].values, w,
                label=res['name'], color=colors_m[j], alpha=0.85)
axes[1].axhline(0.8, color='red', ls='--', alpha=0.5, label='Seuil 0.8')
axes[1].set_xticks(x2+w)
axes[1].set_xticklabels([t.replace('NR-','').replace('SR-','') for t in TARGET_COLS],
                         rotation=45, ha='right', fontsize=8)
axes[1].set_ylim([0.4,1.05]); axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('AUC-ROC par Cible', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/07_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [21]:
# ── Courbes ROC (Random Forest, 12 cibles) ───────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.ravel()
for i, t in enumerate(TARGET_COLS):
    fpr, tpr, _ = roc_curve(y_test[:,i], rf_res['y_proba'][:,i])
    roc_auc = auc(fpr, tpr)
    axes[i].plot(fpr, tpr, '#E91E63', lw=2, label=f'AUC={roc_auc:.3f}')
    axes[i].plot([0,1],[0,1], 'gray', ls='--', lw=1)
    axes[i].fill_between(fpr, tpr, alpha=0.1, color='#E91E63')
    axes[i].set_title(t, fontsize=9, fontweight='bold')
    axes[i].legend(loc='lower right', fontsize=8)
    axes[i].set_xlabel('FPR',fontsize=7); axes[i].set_ylabel('TPR',fontsize=7)
plt.suptitle('Courbes ROC – Random Forest (12 cibles)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/08_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [22]:
# ── Matrices de confusion (Random Forest) ───────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.ravel()
for i, t in enumerate(TARGET_COLS):
    cm = confusion_matrix(y_test[:,i], rf_res['y_pred'][:,i])
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Blues',
                xticklabels=['Négatif','Positif'], yticklabels=['Négatif','Positif'])
    axes[i].set_title(t, fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Prédit',fontsize=7); axes[i].set_ylabel('Réel',fontsize=7)
plt.suptitle('Matrices de Confusion – Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/09_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [23]:
# ── Rapport de classification par cible ─────────────────────────────────────
print('='*65)
print('RAPPORTS DE CLASSIFICATION – RANDOM FOREST')
print('='*65)
for i, t in enumerate(TARGET_COLS):
    print(f'\n── {t} ──')
    print(classification_report(y_test[:,i], rf_res['y_pred'][:,i],
                                 target_names=['Non-toxique','Toxique'], zero_division=0))


RAPPORTS DE CLASSIFICATION – RANDOM FOREST

── NR-AR ──
              precision    recall  f1-score   support

 Non-toxique       0.98      0.99      0.99       596
     Toxique       0.56      0.26      0.36        19

    accuracy                           0.97       615
   macro avg       0.77      0.63      0.67       615
weighted avg       0.96      0.97      0.97       615


── NR-AR-LBD ──
              precision    recall  f1-score   support

 Non-toxique       0.99      0.99      0.99       606
     Toxique       0.56      0.56      0.56         9

    accuracy                           0.99       615
   macro avg       0.77      0.77      0.77       615
weighted avg       0.99      0.99      0.99       615


── NR-AhR ──
              precision    recall  f1-score   support

 Non-toxique       0.95      1.00      0.98       584
     Toxique       0.75      0.10      0.17        31

    accuracy                           0.95       615
   macro avg       0.85      0.55      0.

---
## 7. Analyse du Seuil de Décision


In [24]:
# Optimisation du seuil sur SR-MMP (label le moins déséquilibré)
TARGET_IDX  = TARGET_COLS.index('SR-MMP')
y_true_mmp  = y_test[:, TARGET_IDX]
y_prob_mmp  = rf_res['y_proba'][:, TARGET_IDX]

thresholds = np.arange(0.05, 0.95, 0.05)
f1s, precs, recs = [], [], []
for th in thresholds:
    yp = (y_prob_mmp >= th).astype(int)
    f1s.append(f1_score(y_true_mmp, yp, zero_division=0))
    precs.append(precision_score(y_true_mmp, yp, zero_division=0))
    recs.append(recall_score(y_true_mmp, yp, zero_division=0))

best_th = thresholds[np.argmax(f1s)]
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1s,  'b-o', label='F1', lw=2)
ax.plot(thresholds, precs, 'g--s', label='Précision', lw=1.5)
ax.plot(thresholds, recs,  'r--^', label='Rappel', lw=1.5)
ax.axvline(best_th, color='purple', ls=':', label=f'Seuil optimal F1 = {best_th:.2f}')
ax.set_xlabel('Seuil de décision'); ax.set_ylabel('Score')
ax.set_title('Impact du Seuil – SR-MMP (Random Forest)', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/10_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Seuil optimal (max F1) pour SR-MMP : {best_th:.2f}')
print(f'F1 à seuil 0.50  : {f1s[9]:.4f}')
print(f'F1 à seuil optimal: {max(f1s):.4f}')


Seuil optimal (max F1) pour SR-MMP : 0.30
F1 à seuil 0.50  : 0.4103
F1 à seuil optimal: 0.5614


---
## 8. Explainabilité SHAP


In [25]:
# SHAP sur l'estimateur NR-AR (premier label)
print('Calcul SHAP (100 molecules, ~1 min)...')

estimator_nrar = rf_model.estimators_[0]
X_shap = X_test[:100].astype(np.float64)

explainer = shap.TreeExplainer(
    estimator_nrar,
    data=shap.sample(X_train.astype(np.float64), 50, random_state=RANDOM_STATE),
    feature_perturbation='interventional',
)
shap_values = explainer.shap_values(X_shap, check_additivity=False)

# RF binaire -> liste [class0, class1] ou tableau 3D (n, features, 2)
if isinstance(shap_values, list) and len(shap_values) == 2:
    sv = np.array(shap_values[1])
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    sv = shap_values[:, :, 1]
else:
    sv = np.array(shap_values)
    if sv.ndim == 3:
        sv = sv[:, :, 1]

sv = sv.reshape(X_shap.shape[0], -1)  # garantit 2D (n_samples, n_features)
print(f'SHAP calcule | shape : {sv.shape}')


Calcul SHAP (100 molecules, ~1 min)...
SHAP calcule | shape : (100, 2063)


In [26]:
# Summary plot (top 20 features)
plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_shap, feature_names=FEATURE_NAMES, max_display=20, show=False, plot_type='dot')
plt.title('SHAP – Top 20 features (NR-AR, Androgen Receptor)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/11_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()


In [27]:
# Importance des descripteurs moléculaires
desc_imp = np.abs(sv[:, :15]).mean(axis=0)
desc_imp_df = pd.DataFrame({'Descripteur': DESCRIPTOR_NAMES, 'SHAP': desc_imp}).sort_values('SHAP')
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(desc_imp_df['Descripteur'], desc_imp_df['SHAP'],
        color=plt.cm.viridis(np.linspace(0.15, 0.9, 15)))
ax.set_xlabel('|SHAP| moyen')
ax.set_title('Importance des Descripteurs Moléculaires – NR-AR (SHAP)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/12_shap_descriptor_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Descripteurs les plus importants :')
print(desc_imp_df.sort_values('SHAP', ascending=False).to_string(index=False))


Descripteurs les plus importants :
      Descripteur     SHAP
         NumBonds 0.002410
         NumAtoms 0.002004
            MolMR 0.001826
   NumHeteroatoms 0.001589
            MolWt 0.001518
         NumRings 0.001422
             LogP 0.001411
   HeavyAtomCount 0.001381
   RotatableBonds 0.001343
     FractionCSP3 0.001314
             TPSA 0.001159
   HBondAcceptors 0.000922
    AromaticRings 0.000700
      HBondDonors 0.000681
NumAliphaticRings 0.000608


In [28]:
# Waterfall pour une molécule toxique prédite (NR-AR)
toxic_idx = np.where(rf_res['y_pred'][:, 0] == 1)[0]
if len(toxic_idx) > 0:
    mol_idx = toxic_idx[0]

    # expected_value peut etre un array selon la version de SHAP
    ev_raw = explainer.expected_value
    ev_arr = np.array(ev_raw).ravel()
    # RF binaire : [ev_class0, ev_class1] → prendre class 1
    ev = float(ev_arr[1]) if len(ev_arr) > 1 else float(ev_arr[0])

    shap_exp = shap.Explanation(
        values=sv[mol_idx],
        base_values=ev,
        data=X_shap[mol_idx],
        feature_names=FEATURE_NAMES,
    )
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(shap_exp, max_display=15, show=False)
    plt.title('SHAP Waterfall – Molecule toxique (NR-AR)', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/13_shap_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Waterfall sauvegarde.')
else:
    print('Aucune molecule toxique (NR-AR) dans echantillon SHAP — waterfall ignore.')


Waterfall sauvegarde.


---
## 9. Prédiction sur Nouvelles Molécules


In [29]:
def predict_profile(smiles_list, model, threshold=0.5):
    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rows.append({'SMILES': smi, 'Valide': False}); continue
        desc = compute_descriptors(mol)
        if desc is None:
            rows.append({'SMILES': smi, 'Valide': False}); continue
        X_new = np.hstack([desc, compute_morgan_fp(mol)]).reshape(1,-1).astype(np.float32)
        proba = np.column_stack([est.predict_proba(X_new)[:,1] for est in model.estimators_])[0]
        pred  = (proba >= threshold).astype(int)
        row   = {'SMILES': smi, 'Valide': True, 'N Cibles Toxiques': int(pred.sum())}
        for j, t in enumerate(TARGET_COLS):
            row[f'{t} (prob)'] = round(float(proba[j]), 3)
        rows.append(row)
    return pd.DataFrame(rows)

# ── Molécules de référence ───────────────────────────────────────────────────
TEST_MOLS = {
    'Aspirine':          'CC(=O)Oc1ccccc1C(=O)O',
    'Benzène':           'c1ccccc1',
    'Éthanol':           'CCO',
    'Caféine':           'Cn1cnc2c1c(=O)n(c(=O)n2C)C',
    'Bisphenol A':       'CC(c1ccc(O)cc1)(c1ccc(O)cc1)C',
    'Tamoxifène':        'CCC(=C(c1ccccc1)c1ccc(OCCN(C)C)cc1)c1ccccc1',
    'Paracétamol':       'CC(=O)Nc1ccc(O)cc1',
    'Diéthylstilbestrol':'CC(/C=C/c1ccc(O)cc1)=C(\\c1ccc(O)cc1)/CC',
    'Atrazine':          'CCNc1nc(Cl)nc(NC(C)C)n1',
    'Ibuprofen':         'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
    'TCDD (Dioxine)':    'Clc1cc2c(cc1Cl)Oc1c(Cl)c(Cl)ccc1O2',
    'Méthanol':          'CO',
}

pred_df = predict_profile(list(TEST_MOLS.values()), rf_model)
pred_df.insert(0, 'Molécule', list(TEST_MOLS.keys()))

display_cols = ['Molécule', 'N Cibles Toxiques'] + [f'{t} (prob)' for t in TARGET_COLS]
pred_df[display_cols]


,Molécule,N Cibles Toxiques,NR-AR (prob),NR-AR-LBD (prob),NR-AhR (prob),NR-Aromatase (prob),NR-ER (prob),NR-ER-LBD (prob),NR-PPAR-gamma (prob),SR-ARE (prob),SR-ATAD5 (prob),SR-HSE (prob),SR-MMP (prob),SR-p53 (prob)
0,Aspirine,0,0.063,0.015,0.055,0.005,0.087,0.000,0.015,0.066,0.000,0.082,0.014,0.005
1,Benzène,0,0.043,0.000,0.027,0.020,0.080,0.019,0.000,0.035,0.000,0.000,0.005,0.005
2,Éthanol,0,0.000,0.000,0.005,0.000,0.012,0.005,0.000,0.026,0.000,0.010,0.005,0.000
3,Caféine,1,0.005,0.000,0.019,0.005,0.127,0.000,0.000,0.571,0.000,0.015,0.003,0.000
4,Bisphenol A,1,0.010,0.020,0.199,0.295,0.471,0.330,0.134,0.217,0.005,0.260,0.587,0.213
5,Tamoxifène,0,0.059,0.010,0.060,0.088,0.159,0.033,0.010,0.134,0.015,0.058,0.180,0.020
6,Paracétamol,0,0.014,0.020,0.083,0.005,0.152,0.000,0.000,0.075,0.000,0.000,0.061,0.010
7,Diéthylstilbestrol,1,0.064,0.015,0.078,0.172,0.514,0.364,0.060,0.429,0.015,0.340,0.441,0.283
8,Atrazine,0,0.000,0.000,0.145,0.005,0.026,0.000,0.000,0.067,0.000,0.212,0.018,0.005
9,Ibuprofen,2,0.000,0.000,0.023,0.000,0.632,0.015,0.040,0.000,0.000,0.000,0.630,0.020


In [30]:
# ── Heatmap des profils de toxicité ─────────────────────────────────────────
valid_df   = pred_df[pred_df['Valide'] == True]
prob_mat   = valid_df[[f'{t} (prob)' for t in TARGET_COLS]].values
mol_names  = valid_df['Molécule'].values

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(prob_mat, cmap='RdYlGn_r', vmin=0, vmax=1, aspect='auto')
for i in range(prob_mat.shape[0]):
    for j in range(prob_mat.shape[1]):
        v = prob_mat[i,j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                color='white' if v>0.5 else 'black', fontweight='bold')
ax.set_xticks(range(len(TARGET_COLS)))
ax.set_xticklabels([t.replace('NR-','').replace('SR-','') for t in TARGET_COLS], rotation=45, ha='right')
ax.set_yticks(range(len(mol_names))); ax.set_yticklabels(mol_names)
plt.colorbar(im, ax=ax, label='Probabilité de toxicité')
ax.set_title('Profil de Toxicité Multi-Label – Molécules de Référence (RF)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/14_toxicity_profiles.png', dpi=150, bbox_inches='tight')
plt.show()


In [31]:
# ── Radar chart ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(18, 13), subplot_kw=dict(polar=True))
axes = axes.ravel()
angles = np.linspace(0, 2*np.pi, len(TARGET_COLS), endpoint=False).tolist()
colors_r = plt.cm.tab10(np.linspace(0, 1, len(mol_names)))

for k, (name, color) in enumerate(zip(mol_names, colors_r)):
    p = prob_mat[k].tolist()
    a = angles + angles[:1]; pp = p + p[:1]
    axes[k].plot(a, pp, color=color, lw=2)
    axes[k].fill(a, pp, color=color, alpha=0.2)
    axes[k].set_xticks(angles)
    axes[k].set_xticklabels([t.replace('NR-','').replace('SR-','') for t in TARGET_COLS], fontsize=7)
    axes[k].set_ylim(0, 1)
    axes[k].set_title(name, size=9, fontweight='bold', pad=12)

for j in range(k+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Radar – Probabilités de Toxicité Multi-Label', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/15_toxicity_radar.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 10. Sauvegarde du Modèle


In [32]:
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
package = {
    'model':   rf_model,
    'targets': TARGET_COLS,
    'feature_engineering': {
        'descriptor_names':  DESCRIPTOR_NAMES,
        'n_descriptors':     len(DESCRIPTOR_NAMES),
        'fp_radius':         FP_RADIUS,
        'fp_nbits':          FP_NBITS,
        'n_features_total':  len(DESCRIPTOR_NAMES) + FP_NBITS,
    },
    'metrics': {
        'hamming':   rf_res['hamming'],   'jaccard':   rf_res['jaccard'],
        'f1_micro':  rf_res['f1_micro'],  'f1_macro':  rf_res['f1_macro'],
        'auc_macro': rf_res['auc_macro'],
        'per_label_auc': rf_res['per_label']['AUC-ROC'].to_dict(),
    },
    'training_info': {
        'n_train': X_train.shape[0], 'n_test': X_test.shape[0],
        'n_features': X.shape[1],   'n_labels': len(TARGET_COLS),
    },
}
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(package, f)

print(f'Modele sauvegarde : {MODEL_PATH}')
print(f'   AUC-ROC macro : {rf_res["auc_macro"]}  |  F1 macro : {rf_res["f1_macro"]}  |  Hamming : {rf_res["hamming"]}')


Modele sauvegarde : c:\Users\oumaimaaa\Desktop\master big data\S2 BDDS\Machine learning\PROJET MACHING LEARNING\models\multilabel_rf_model.pkl
   AUC-ROC macro : 0.7833  |  F1 macro : 0.2371  |  Hamming : 0.0305


In [33]:
# ── Exemple d'utilisation du modèle chargé ───────────────────────────────────
with open(MODEL_PATH, 'rb') as f:
    loaded = pickle.load(f)

result = predict_profile(['CC(=O)Oc1ccccc1C(=O)O'], loaded['model'])
print('Test – Aspirine :')
print(f'  N cibles toxiques prédites : {result["N Cibles Toxiques"].values[0]}')
print('✅ Modèle chargé et opérationnel')


Test – Aspirine :
  N cibles toxiques prédites : 0
✅ Modèle chargé et opérationnel


---
## 11. Conclusions

### Résultats clés

| Aspect | Résultat |
|--------|----------|
| **Dataset** | 3 000+ molécules avec les 12 labels complets |
| **Features** | 15 descripteurs RDKit + 2048 bits ECFP4 = 2063 features |
| **Modèle retenu** | Random Forest (200 arbres × 12 cibles, `class_weight='balanced'`) |
| **Déséquilibre** | Géré par pondération + optimisation du seuil |
| **Explainabilité** | LogP, MolWt et TPSA sont les descripteurs les plus influents (NR-AR) |

### Limites
- ~3 000 molécules : dataset limité → overfitting possible sur les cibles rares
- Les 12 classifieurs sont indépendants (pas de dépendances entre labels capturées)
- Morgan fingerprints ne capturent pas la 3D moléculaire

### Perspectives
- **Graph Neural Networks (GNN)** : représentation moléculaire complète
- **Classifier Chains** : capturer les corrélations entre labels
- **Données externes** : augmentation avec ChEMBL / PubChem
- **Application web** : Streamlit pour la prédiction interactive
